In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
%load_ext autoreload
%autoreload 2

# Model definition

In [3]:
import hydra
from omegaconf import OmegaConf
import torch

conf = OmegaConf.load('config/age_whisper.yaml')

# Finetune

In [4]:
from glob import glob
from ptls.data_load.iterable_processing_dataset import IterableProcessingDataset
from ptls.data_load.iterable_processing.feature_filter import FeatureFilter
from ptls.data_load.iterable_processing.to_torch_tensor import ToTorch
from ptls.data_load.datasets import MemoryMapDataset
from ptls.frames.supervised import SeqToTargetDataset
from tqdm.auto import tqdm
from ptls.data_load.iterable_processing.target_empty_filter import TargetEmptyFilter

from ptls.data_load import IterableChain
from ptls.data_load.iterable_processing import SeqLenFilter, ISeqLenLimit
from ptls.data_load.datasets.parquet_dataset import ParquetDataset, ParquetFiles
from ptls.data_load.utils import collate_feature_dict
from ptls.frames import PtlsDataModule

train_data = glob('/home/morlov/ptls-experiments/scenario_age_pred/data/train_trx_file.parquet')
valid_data = glob('/home/morlov/ptls-experiments/scenario_age_pred/data/test_trx_file.parquet')

feature_cols = list(conf.seq_encoder.trx_encoder.embeddings.keys()) + \
               list(conf.seq_encoder.trx_encoder.numeric_values.keys())

dataset_conf = {
    'min_seq_len':25,
    'max_seq_len':500,
    'drop_feature_names': ['client_id', 'event_time', 'trx_count', 'trans_date'],
    'target_col': 'target'
    }


process = IterableChain(
            SeqLenFilter(min_seq_len=dataset_conf['min_seq_len']),
            ISeqLenLimit(max_seq_len=dataset_conf['max_seq_len']),
            TargetEmptyFilter(dataset_conf['target_col']),
            FeatureFilter(keep_feature_names=feature_cols + ['target']),
            ToTorch()
            )
    
def get_dataset(data):
    ds = MemoryMapDataset(ParquetDataset(data, post_processing=process))
    return SeqToTargetDataset(ds, target_col_name=dataset_conf['target_col'])

train_ds = get_dataset(train_data)
valid_ds = get_dataset(valid_data)

dm = PtlsDataModule(
    train_data=train_ds,
    valid_data=valid_ds,
    train_num_workers=4,
    train_batch_size=64)

[2024-08-07 08:41:16,920] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/morlov/.local/share/virtualenvs/pytorch-lifestream-1iBTwtzi/lib/python3.8/site-packages/ptls/data_load/datasets/parquet_dataset.py:106: UserWarning: `post_processing` parameter is deprecated, use `i_filters`
  warnings.warn('`post_processing` parameter is deprecated, use `i_filters`')


In [5]:
from ptls.nn import TrxEncoder

trx_encoder_params = conf['seq_encoder']['trx_encoder']
trx_encoder = TrxEncoder(**trx_encoder_params)

In [6]:
from ptls.nn import RnnEncoder

seq_encoder_params = {'type': 'gru', 'hidden_size': 800, 'bidir': False}
rnn_seq_encoder = RnnEncoder(input_size=768, is_reduce_sequence=True, **seq_encoder_params)

In [7]:
from romashka.transactions_qa.layers.layers import LambdaLayer
from romashka.transactions_qa.utils import zero_function
from ptls.data_load.padded_batch import PaddedBatch

class WhisperSeqEncoder(torch.nn.Module):
    
    def __init__(self, trx_encoder, seq_encoder, whisper_encoder):
        super().__init__()
        self.trx_encoder = trx_encoder
        self.seq_encoder = seq_encoder
        self.whisper_encoder = whisper_encoder
        
    @property
    def is_reduce_sequence(self):
        return self.seq_encoder.is_reduce_sequence

    @is_reduce_sequence.setter
    def is_reduce_sequence(self, value):
        self.seq_encoder.is_reduce_sequence = value

    @property
    def category_max_size(self):
        return self.trx_encoder.category_max_size

    @property
    def category_names(self):
        return self.trx_encoder.category_names

    @property
    def embedding_size(self):
        return self.seq_encoder.embedding_size

    def forward(self, x, h_0=None):
        x = self.trx_encoder(x)
        length = x._length
        x = self.whisper_encoder(inputs_embeds=x.payload, attention_mask=x.seq_len_mask).last_hidden_state
        x = PaddedBatch(payload=x, length=length)
        x = self.seq_encoder(x, h_0)
        return x
    
from transformers import AutoModel, AutoConfig


whisper_encoder = AutoModel.from_pretrained('openai/whisper-small').decoder
whisper_encoder.embed_positions = LambdaLayer(zero_function)
seq_encoder = WhisperSeqEncoder(trx_encoder, rnn_seq_encoder, whisper_encoder)

In [8]:
with torch.no_grad():
    valid_dl = torch.utils.data.DataLoader(dataset=valid_ds, collate_fn=valid_ds.collate_fn, num_workers=8, batch_size=4)
    batch = next(iter(valid_dl))
    seq_encoder(batch[0])

In [9]:
from functools import partial
import torch
import torchmetrics
from ptls.frames.supervised import SequenceToTarget
from ptls.nn import Head

model_e2e = SequenceToTarget(
    seq_encoder=seq_encoder,
    head=Head(
        input_size=seq_encoder.embedding_size,
        use_batch_norm=True,
        objective='classification',
        num_classes=4,
    ),
    loss=torch.nn.NLLLoss(),
    metric_list=torchmetrics.Accuracy(compute_on_step=False),
    pretrained_lr=3e-4,
    optimizer_partial=partial(torch.optim.Adam, lr=3e-4, weight_decay=1e-5),
    lr_scheduler_partial=partial(torch.optim.lr_scheduler.StepLR, step_size=10, gamma=0.9),
)


In [10]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

trainer_params = conf.trainer


trainer_params = conf.trainer
trainer_params['max_epochs'] = 10
callbacks = [ModelCheckpoint(every_n_epochs=1, save_top_k=-1), LearningRateMonitor(logging_interval='step')]
logger = TensorBoardLogger(save_dir='lightning_logs', name='whisper')

print(OmegaConf.to_yaml(trainer_params))

trainer = pl.Trainer(**trainer_params, callbacks=callbacks, logger=logger)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


gpus: 1
auto_select_gpus: false
max_epochs: 10
deterministic: true



In [11]:
%%time
print(f'logger.version = {trainer.logger.version}')
trainer.fit(model_e2e, dm)
print(trainer.logged_metrics)

logger.version = 11


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type              | Params
----------------------------------------------------
0 | seq_encoder   | WhisperSeqEncoder | 157 M 
1 | head          | Head              | 4.8 K 
2 | loss          | NLLLoss           | 0     
3 | train_metrics | ModuleDict        | 0     
4 | valid_metrics | ModuleDict        | 0     
5 | test_metrics  | ModuleDict        | 0     
----------------------------------------------------
157 M     Trainable params
0         Non-trainable params
157 M     Total params
628.213   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

{'loss': tensor(0.7385, device='cuda:0'), 'seq_len': tensor(500., device='cuda:0'), 'y': tensor(1.6094, device='cuda:0'), 'val_loss': tensor(1.9439, device='cuda:0'), 'val_Accuracy': tensor(0.3403, device='cuda:0'), 'train_Accuracy': tensor(0.6021, device='cuda:0')}
CPU times: user 1h 25min 32s, sys: 16min 49s, total: 1h 42min 22s
Wall time: 1h 29min 51s


/home/morlov/.local/share/virtualenvs/pytorch-lifestream-1iBTwtzi/lib/python3.8/site-packages/pytorch_lightning/trainer/trainer.py:726: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")


In [12]:
torch.save(model_e2e.state_dict(), "models/whisper-age-e2e-pd.pt")

# Infernece

In [13]:
model_e2e.load_state_dict(torch.load("models/whisper-age-e2e-pd.pt"))

<All keys matched successfully>

In [14]:
# %%time
import tqdm

def inference(model, dl, device='cuda:0'):
    
    model.to(device)
    X = []
    for batch in tqdm.tqdm(dl):
        with torch.no_grad():
            features = batch[0]
            targets = [batch[1].to(device).unsqueeze(dim=1)]
            x = model(features.to(device))
            flag = torch.argmax(x, dim=1).unsqueeze(dim=1)
            predicted = [flag]
            X += [torch.cat(predicted + targets, dim=1)]
    return X


valid_dl = torch.utils.data.DataLoader(dataset=valid_ds, 
                                       collate_fn=valid_ds.collate_fn,
                                       num_workers=8,
                                       batch_size=64)

In [15]:
preds = torch.vstack(inference(model_e2e, valid_dl)).cpu().numpy()


100%|███████████████████████████████████████████| 47/47 [00:33<00:00,  1.39it/s]


In [16]:
import numpy as np

df_valid = pd.DataFrame(preds, columns = ['predicted_flag', 'flag'])
df_valid.head()

,predicted_flag,flag
0,0,0
1,3,3
2,2,2
3,3,0
4,0,0


In [17]:
from sklearn.metrics import accuracy_score

print("Roc AUC score:", {accuracy_score(df_valid['flag'],  df_valid['predicted_flag'])})

Roc AUC score: {0.5686666666666667}
